### Deterministic Guardrails

In [ ]:
test_inputs = [
    "Is it legal to Attack a server?",
    "How do I defend against a virus?",
    "I got an email that looks like a phish",
    "He carried a kitchen knife",      # should match "knife"
    "The attacker ran away",           # should match "attack" as a word
    "I love phish music",              # should not match "phish" if treated as a band name but still matches naive keyword
]

banned_keywords = ["attack", "phish", "virus", "knife"]

for sentence in test_inputs: # iter 1 --> sentence : Is it legal to Attack a server?
  for kw in banned_keywords:
    if kw in sentence.lower():
      print(f"the word {kw} exist in --> {sentence} -- Result: ",True)
      # print()
# for kw in banned_keywords:
#   print(kw)
  # print(any(kw in test_inputs.lower()))
  # for index,value in enumerate(test_inputs):
  #   if kw in test_inputs[index].lower():
  #     print(f"the word {kw} exist in --> {test_inputs[index]} ")

  # print(kw)

the word attack exist in --> Is it legal to Attack a server? -- Result:  True
the word virus exist in --> How do I defend against a virus? -- Result:  True
the word phish exist in --> I got an email that looks like a phish -- Result:  True
the word knife exist in --> He carried a kitchen knife -- Result:  True
the word attack exist in --> The attacker ran away -- Result:  True
the word phish exist in --> I love phish music -- Result:  True


In [20]:
# for sentence in test_inputs:
any(test_inputs)
  # for kw in banned_keywords:
  #   if kw in sentence.lower():
  #     print(f"the word {kw} exist in --> {sentence} ")

True

In [5]:
import re

# def deterministic_guardrail(text: str) -> bool: # text = inp, text = Is it legal to attack a server?
#     """Returns True if content is blocked."""
#     banned_keywords = ["attack", "phish", "virus", "knife"]
#     return any(kw in text.lower() for kw in banned_keywords)

def deterministic_guardrail_re(text: str) -> bool: # text = inp, text = Is it legal to attack a server?
    """Returns True if content is blocked, using regex word boundaries."""
    banned = r"\b(?:attack|phish|virus|knife)\b"
    return bool(re.search(banned, text, flags=re.IGNORECASE))

test_inputs = [
    "Is it legal to Attack a server?",
    "How do I defend against a virus?",
    "I got an email that looks like a phish",
    "He carried a kitchen knife",      # should match "knife"
    "The attacker ran away",           # should match "attack" as a word
    "I love phish music",              # should not match "phish" if treated as a band name but still matches naive keyword
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs: # iteration1 --> inp = Is it legal to attack a server?
    blocked = deterministic_guardrail_re(inp)
    # blocked = deterministic_guardrail(inp) # blocked value could be either true or false
    status = "⛔ BLOCKED" if blocked else "✅ ALLOWED" # if block value is true
    print(f"{status}  {inp}")

=== Deterministic Guardrail Demo ===
⛔ BLOCKED  Is it legal to Attack a server?
⛔ BLOCKED  How do I defend against a virus?
⛔ BLOCKED  I got an email that looks like a phish
⛔ BLOCKED  He carried a kitchen knife
✅ ALLOWED  The attacker ran away
⛔ BLOCKED  I love phish music


### LLM Based Guardrails

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
# from dotenv import load_dotenv, find_dotenv
# load_dotenv(find_dotenv(usecwd=True))

True

In [3]:

from openai import OpenAI
import os
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

response = client.responses.create(
    input="Explain the importance AI in 2 lines",
    model="openai/gpt-oss-20b",
)
print(response.output_text)


Artificial Intelligence accelerates innovation, automates complex tasks, and uncovers insights across sectors. It drives economic growth, improves decision‑making, and expands human capabilities.


In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

def model_based_guardrail(text: str) -> str: # text = Is it legal to Attack a server?
    prompt = (
        "Classify this user input as SAFE or UNSAFE. "
        "Reply with only one word: ✅ SAFE or ⛔ UNSAFE.\n\n"
        f"Input: {text}"
    )
    response = llm.invoke(prompt)
    print("raw response",response.content)
    return response.content.strip().upper()

print("=== LLM (groq) Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp) # inp = Is it legal to Attack a server?
    print(f"{verdict}: {inp}")

=== LLM (groq) Guardrail Demo ===
raw response ⛔ UNSAFE
⛔ UNSAFE: Is it legal to Attack a server?
raw response ✅ SAFE
✅ SAFE: How do I defend against a virus?
raw response ⛔ UNSAFE
⛔ UNSAFE: I got an email that looks like a phish
raw response ⛔ UNSAFE
⛔ UNSAFE: He carried a kitchen knife
raw response ✅ SAFE
✅ SAFE: The attacker ran away
raw response ✅ SAFE
✅ SAFE: I love phish music


##### why return response.content.strip().upper()?

`llm.invoke(prompt)` returns an AI message object, not a plain string. `response.content` pulls out the actual text the model replied with.

`strip()` removes leading/trailing whitespace or newline noise, and `upper()` normalizes variants like `safe`, `SAFE`, or `Safe` into one consistent value: `SAFE` or `UNSAFE`.

So this line:

```python
return response.content.strip().upper()
```

is basically making the guardrail output easier and safer to compare later.

##### Reference
https://docs.langchain.com/oss/python/integrations/chat/groq  


https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq

In [9]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

def model_based_guardrail(text: str) -> str:
    prompt = (
        "Classify this user input as SAFE or UNSAFE. "
        "Reply with only one word: ✅ SAFE or ⛔ UNSAFE.\n\n"
        f"Input: {text}"
    )
    response = client.responses.create(
        model="llama-3.1-8b-instant",
        input=prompt,
    )
    return response.output_text.strip().upper()

print("=== LLM (groq) Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    print(f"{verdict}: {inp}")

=== LLM (groq) Guardrail Demo ===
⛔ UNSAFE: Is it legal to Attack a server?
✅ SAFE: How do I defend against a virus?
✅ SAFE: I got an email that looks like a phish
⛔ UNSAFE: He carried a kitchen knife
✅ SAFE: The attacker ran away
✅ SAFE: I love phish music
